In [1]:
import cv2
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# --------------------------
# CONFIG: Give folder containing input images
# --------------------------
input_folder = r"D:\projects\neuro\all fascicles\fascicle images\set2_Image_01_20x_bf_05_all"
output_root = r"C:\Users\DELL\Downloads\20x_bf_05_all_thickness"
os.makedirs(output_root, exist_ok=True)

# Scale factor (µm/px)
scale_factor = 0.136

# --------------------------
# Ray casting config
# --------------------------
N_RAYS = 36          # Number of rays fired (every 10 degrees)
RAY_STEP = 0.5       # Step size in pixels along each ray

# --------------------------
# Tunables
# ---------------------------
TISSUE_OVERLAP_MIN = 0.50
MAX_MEAN_INTENSITY = 245

MIN_CIRCULARITY = 0.05
MIN_CONTOUR_AREA = 8
MAX_CONTOUR_AREA = 2000000
MIN_SOLIDITY = 0.15

# Precompute ray angle labels once (used for column headers)
RAY_ANGLES = np.linspace(0, 360, N_RAYS, endpoint=False)
RAY_COL_LABELS = [f"ray_{int(a):03d}deg_um" for a in RAY_ANGLES]

# --------------------------
# Helpers
# --------------------------
def darken_and_sharpen(image):
    darkened = cv2.convertScaleAbs(image, alpha=1.5, beta=-20)
    sharpen_kernel = np.array([[0, -1, 0],
                               [-1, 5, -1],
                               [0, -1, 0]])
    return cv2.filter2D(darkened, -1, sharpen_kernel)


def build_tissue_mask(full_image):
    hsv = cv2.cvtColor(full_image, cv2.COLOR_BGR2HSV)
    H, S, V = cv2.split(hsv)
    sat_mask = cv2.inRange(S, 15, 255)
    not_too_bright = cv2.inRange(V, 0, 250)
    mask = cv2.bitwise_and(sat_mask, not_too_bright)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k, iterations=2)
    return mask


def contour_overlap_ratio(contour, mask):
    c_mask = np.zeros(mask.shape, dtype=np.uint8)
    cv2.drawContours(c_mask, [contour], -1, 255, -1)
    inter = cv2.bitwise_and(c_mask, mask)
    area_c = cv2.countNonZero(c_mask)
    area_i = cv2.countNonZero(inter)
    return area_i / float(area_c) if area_c > 0 else 0.0


def mean_intensity_in_contour(gray, contour):
    m = np.zeros(gray.shape, dtype=np.uint8)
    cv2.drawContours(m, [contour], -1, 255, -1)
    return cv2.mean(gray, mask=m)[0]


def circularity(contour):
    a = cv2.contourArea(contour)
    p = cv2.arcLength(contour, True)
    return 4 * np.pi * a / (p * p + 1e-6) if p > 0 else 0


# --------------------------
# Ray Casting
# --------------------------
def cast_rays_to_boundary(contour, cx, cy, n_rays=N_RAYS, step=RAY_STEP):
    """
    Fire n_rays from (cx, cy) outward in equally spaced directions.
    For each ray, walks outward until it exits the filled contour.

    Returns:
        ray_distances_px : np.array of shape (n_rays,), distances in pixels
        ray_angles_deg   : np.array of shape (n_rays,), corresponding angles
    """
    # Build a local binary mask big enough to contain the contour
    x_b, y_b, w_b, h_b = cv2.boundingRect(contour)
    pad = 20
    max_r = int(max(w_b, h_b) + pad)

    # Shift contour so that (cx, cy) maps to (max_r, max_r) in local coords
    local_cx = max_r
    local_cy = max_r
    shifted = contour - np.array([[int(cx) - max_r, int(cy) - max_r]])

    mask_size = max_r * 2 + 2
    local_mask = np.zeros((mask_size, mask_size), dtype=np.uint8)
    cv2.drawContours(local_mask, [shifted], -1, 255, -1)

    angles = np.linspace(0, 2 * np.pi, n_rays, endpoint=False)
    ray_distances_px = np.zeros(n_rays)

    for i, angle in enumerate(angles):
        cos_a = np.cos(angle)
        sin_a = np.sin(angle)

        # Walk outward along this direction
        r_vals = np.arange(0, max_r, step)
        xs = (local_cx + r_vals * cos_a).astype(int)
        ys = (local_cy + r_vals * sin_a).astype(int)

        # Clip to mask bounds
        valid = (xs >= 0) & (xs < mask_size) & (ys >= 0) & (ys < mask_size)
        xs_v = xs[valid]
        ys_v = ys[valid]
        r_v  = r_vals[valid]

        if len(xs_v) == 0:
            ray_distances_px[i] = 0
            continue

        pixel_vals = local_mask[ys_v, xs_v]

        # Find first index where we go from inside (255) to outside (0)
        inside_idx = np.where(pixel_vals == 255)[0]
        if len(inside_idx) == 0:
            # centre not inside contour – unusual, fall back to 0
            ray_distances_px[i] = 0
            continue

        outside_after = np.where((pixel_vals == 0) & (np.arange(len(pixel_vals)) > inside_idx[0]))[0]
        if len(outside_after) > 0:
            ray_distances_px[i] = r_v[outside_after[0]]
        else:
            # ray never exited – use the last valid r
            ray_distances_px[i] = r_v[-1]

    return ray_distances_px, np.degrees(angles)


def draw_rays_on_image(output_image, cx, cy, ray_distances_px, ray_angles_deg, chosen_idx, x_offset, y_offset):
    """Draw all rays in cyan, the shortest (chosen) ray in red."""
    angles_rad = np.deg2rad(ray_angles_deg)
    for i, (r, ang) in enumerate(zip(ray_distances_px, angles_rad)):
        end_x = int(cx + r * np.cos(ang))
        end_y = int(cy + r * np.sin(ang))
        color = (0, 0, 255) if i == chosen_idx else (200, 200, 0)  # red vs dark-cyan
        cv2.line(output_image,
                 (int(cx + x_offset), int(cy + y_offset)),
                 (end_x + x_offset, end_y + y_offset),
                 color, 1)


# --------------------------
# Main patch processing
# --------------------------
def process_patch(patch, mask_patch, x_offset, y_offset, axon_data, object_counter, output_image):

    patch = darken_and_sharpen(patch)
    gray = cv2.cvtColor(patch, cv2.COLOR_BGR2GRAY)

    smooth = cv2.bilateralFilter(gray, 5, 20, 20)

    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    enhanced = clahe.apply(smooth)

    blurred = cv2.GaussianBlur(enhanced, (3, 3), 0)
    _, otsu = cv2.threshold(blurred, 0, 255,
                            cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    adaptive = cv2.adaptiveThreshold(
        blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, 35, 2
    )

    thresh = cv2.bitwise_or(otsu, adaptive)

    kernel = np.ones((3, 3), np.uint8)
    opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=1)
    sure_bg = cv2.dilate(opening, kernel, iterations=3)

    dist_transform = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
    if dist_transform.max() > 0:
        _, sure_fg = cv2.threshold(dist_transform,
                                   0.4 * dist_transform.max(), 255, 0)
    else:
        sure_fg = np.zeros_like(opening)
    sure_fg = np.uint8(sure_fg)

    unknown = cv2.subtract(sure_bg, sure_fg)
    num_markers, markers = cv2.connectedComponents(sure_fg)
    markers = markers + 1
    markers[unknown == 255] = 0

    wshed_input = cv2.cvtColor(enhanced, cv2.COLOR_GRAY2BGR)
    markers = cv2.watershed(wshed_input, markers)

    cleaned = thresh.copy()
    cleaned[markers == -1] = 0

    contours, hierarchy = cv2.findContours(
        cleaned, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE
    )
    if hierarchy is None:
        return object_counter
    hierarchy = hierarchy[0]

    for i, contour in enumerate(contours):

        if hierarchy[i][3] != -1:
            continue

        area = cv2.contourArea(contour)
        if area < MIN_CONTOUR_AREA or area > MAX_CONTOUR_AREA:
            continue

        overlap = contour_overlap_ratio(contour, mask_patch)
        if overlap < TISSUE_OVERLAP_MIN:
            continue

        hull = cv2.convexHull(contour)
        solidity = area / (cv2.contourArea(hull) + 1e-6)
        circ = circularity(contour)
        if solidity < MIN_SOLIDITY or circ < MIN_CIRCULARITY:
            continue

        # Inner contours
        inner_children = []
        for j, child in enumerate(contours):
            if hierarchy[j][3] == i and cv2.contourArea(child) > 30:
                inner_children.append(child)

        if len(inner_children) == 0:
            continue

        axon_type = "mature" if len(inner_children) <= 2 else "regrowth_cluster"

        (x_outer, y_outer), outer_radius_enc = cv2.minEnclosingCircle(contour)
        contour_shifted = contour + np.array([x_offset, y_offset])
        x_outer_abs = x_outer + x_offset
        y_outer_abs = y_outer + y_offset

        outer_area_px  = cv2.contourArea(contour)
        outer_area_um2 = outer_area_px * (scale_factor ** 2)

        outer_color = (255, 0, 0) if axon_type == "mature" else (0, 255, 255)
        cv2.drawContours(output_image, [contour_shifted], -1, outer_color, 1)

        inner_radii = []
        inner_areas  = []
        for inner_c in inner_children:
            (_, _), ir = cv2.minEnclosingCircle(inner_c)
            inner_shifted = inner_c + np.array([x_offset, y_offset])
            cv2.drawContours(output_image, [inner_shifted], -1, (0, 255, 0), 1)
            inner_radii.append(ir)
            inner_areas.append(cv2.contourArea(inner_c))

        inner_radius   = max(inner_radii)
        inner_area_px  = max(inner_areas)
        inner_area_um2 = inner_area_px * (scale_factor ** 2)

        # --------------------------------------------------
        # RAY CASTING  (mature axons only)
        # --------------------------------------------------
        ray_distances_um_dict = {col: np.nan for col in RAY_COL_LABELS}
        chosen_radius_um      = outer_radius_enc * scale_factor   # default = enclosing circle
        chosen_ray_angle      = np.nan
        all_rays_px           = None

        if axon_type == "mature":
            ray_distances_px, ray_angles_deg = cast_rays_to_boundary(
                contour, x_outer, y_outer, n_rays=N_RAYS, step=RAY_STEP
            )

            # Convert to µm and store per-angle columns
            ray_distances_um = ray_distances_px * scale_factor
            for col_label, dist_um in zip(RAY_COL_LABELS, ray_distances_um):
                ray_distances_um_dict[col_label] = round(float(dist_um), 4)

            # Shortest ray → chosen radius
            chosen_idx       = int(np.argmin(ray_distances_um))
            chosen_radius_um = float(ray_distances_um[chosen_idx])
            chosen_ray_angle = float(ray_angles_deg[chosen_idx])
            all_rays_px      = ray_distances_px

            # Draw rays on output image (local patch coordinates adjusted by offset)
            draw_rays_on_image(
                output_image,
                x_outer, y_outer,
                ray_distances_px, ray_angles_deg,
                chosen_idx,
                x_offset, y_offset
            )

        # Thickness: chosen outer radius minus largest inner radius
        inner_radius_um = inner_radius * scale_factor
        outer_radius_um = outer_radius_enc * scale_factor
        thickness_um    = max(chosen_radius_um - inner_radius_um, 0.0)

        g_ratio    = inner_radius / (outer_radius_enc + 1e-6)
        area_ratio = inner_area_um2 / (outer_area_um2 + 1e-6)

        # Base record
        record = {
            "axon_id"           : object_counter,
            "axon_type"         : axon_type,
            "num_inner_contours": len(inner_children),
            "center_x_px"       : round(float(x_outer_abs), 2),
            "center_y_px"       : round(float(y_outer_abs), 2),
            "outer_radius_um"   : round(outer_radius_um, 4),
            "inner_radius_um"   : round(inner_radius_um, 4),
            "chosen_radius_um"  : round(chosen_radius_um, 4),
            "chosen_ray_angle_deg": round(chosen_ray_angle, 2) if axon_type == "mature" else np.nan,
            "thickness_um"      : round(thickness_um, 4),
            "diameter_um"       : round(2 * outer_radius_um, 4),
            "outer_area_um2"    : round(outer_area_um2, 4),
            "inner_area_um2"    : round(inner_area_um2, 4),
            "area_ratio"        : round(area_ratio, 4),
            "g_ratio"           : round(g_ratio, 4),
        }

        # Merge ray columns (only populated for mature)
        record.update(ray_distances_um_dict)

        axon_data.append(record)

        cv2.putText(output_image, str(object_counter),
                    (int(x_outer_abs), int(y_outer_abs)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)

        object_counter += 1

    return object_counter


# --------------------------
# Whole-image processing
# --------------------------
def process_image(image_path, patch_size=1024):
    image = cv2.imread(image_path)
    if image is None:
        print(f"❌ Cannot read image: {image_path}")
        return None, None

    tissue_mask_full = build_tissue_mask(image)
    h, w = image.shape[:2]

    output_image  = image.copy()
    axon_data     = []
    object_counter = 1

    for y in range(0, h, patch_size):
        for x in range(0, w, patch_size):
            patch      = image[y:y + patch_size, x:x + patch_size]
            mask_patch = tissue_mask_full[y:y + patch_size, x:x + patch_size]

            object_counter = process_patch(
                patch, mask_patch, x, y, axon_data, object_counter, output_image
            )

    print(f"  → Processed {len(axon_data)} axons total "
          f"({sum(1 for a in axon_data if a['axon_type'] == 'mature')} mature, "
          f"{sum(1 for a in axon_data if a['axon_type'] == 'regrowth_cluster')} regrowth).")
    return axon_data, output_image


# --------------------------
# Excel export (styled)
# --------------------------
def save_to_excel(axon_data, output_folder, image_name):
    if not axon_data:
        print("  ⚠ No axon data to save.")
        return None

    df = pd.DataFrame(axon_data)

    # ── Column order ──────────────────────────────────────────────────────────
    base_cols = [
        "axon_id", "axon_type", "num_inner_contours",
        "center_x_px", "center_y_px",
        "outer_radius_um", "inner_radius_um",
        # --- ray columns for mature ---
        *RAY_COL_LABELS,
        # --- summary ---
        "chosen_ray_angle_deg", "chosen_radius_um",
        "thickness_um", "diameter_um",
        "outer_area_um2", "inner_area_um2",
        "area_ratio", "g_ratio",
    ]
    # keep only columns that actually exist in df
    ordered_cols = [c for c in base_cols if c in df.columns]
    df = df[ordered_cols]

    excel_path = os.path.join(output_folder, f"{image_name}_axon_measurements.xlsx")
    df.to_excel(excel_path, index=False, sheet_name="Axon Measurements")

    # ── Styling with openpyxl ─────────────────────────────────────────────────
    wb = openpyxl.load_workbook(excel_path)
    ws = wb.active

    # Colour palettes
    HEADER_FILL    = PatternFill("solid", fgColor="1F4E79")   # dark blue
    RAY_FILL       = PatternFill("solid", fgColor="E8F4FD")   # light blue
    SUMMARY_FILL   = PatternFill("solid", fgColor="FFF2CC")   # light yellow
    MATURE_FILL    = PatternFill("solid", fgColor="D9F0D3")   # light green
    REGROWTH_FILL  = PatternFill("solid", fgColor="FCE4D6")   # light orange
    CHOSEN_FILL    = PatternFill("solid", fgColor="FFD700")   # gold  – chosen radius col
    THICK_FILL     = PatternFill("solid", fgColor="F4CCCC")   # light red – thickness col

    WHITE_FONT     = Font(color="FFFFFF", bold=True, name="Calibri", size=10)
    HEADER_BORDER  = Border(
        bottom=Side(style="medium", color="FFFFFF")
    )
    CELL_BORDER    = Border(
        bottom=Side(style="thin", color="D0D0D0")
    )

    # Identify column indices by name
    col_names = [ws.cell(1, c).value for c in range(1, ws.max_column + 1)]

    def col_idx(name):
        try:
            return col_names.index(name) + 1
        except ValueError:
            return None

    ray_col_indices      = [col_idx(c) for c in RAY_COL_LABELS if col_idx(c)]
    chosen_radius_col    = col_idx("chosen_radius_um")
    chosen_angle_col     = col_idx("chosen_ray_angle_deg")
    thickness_col        = col_idx("thickness_um")
    axon_type_col        = col_idx("axon_type")

    # ── Header row ────────────────────────────────────────────────────────────
    for cell in ws[1]:
        cell.fill      = HEADER_FILL
        cell.font      = WHITE_FONT
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border    = HEADER_BORDER

    # Special fills for chosen-radius and thickness headers
    if chosen_radius_col:
        ws.cell(1, chosen_radius_col).fill = PatternFill("solid", fgColor="B8860B")
    if thickness_col:
        ws.cell(1, thickness_col).fill = PatternFill("solid", fgColor="8B0000")
        ws.cell(1, thickness_col).font = Font(color="FFFFFF", bold=True, name="Calibri", size=10)

    # ── Data rows ─────────────────────────────────────────────────────────────
    for row_idx in range(2, ws.max_row + 1):
        atype = ws.cell(row_idx, axon_type_col).value if axon_type_col else ""
        row_fill = MATURE_FILL if atype == "mature" else REGROWTH_FILL

        for col_idx_ in range(1, ws.max_column + 1):
            cell = ws.cell(row_idx, col_idx_)
            cell.border    = CELL_BORDER
            cell.alignment = Alignment(horizontal="center", vertical="center")

            # Default row colour based on axon type
            cell.fill = row_fill

            # Override for ray columns → light blue
            if col_idx_ in ray_col_indices:
                cell.fill = RAY_FILL

            # Override for chosen radius column → gold
            if chosen_radius_col and col_idx_ == chosen_radius_col:
                cell.fill = CHOSEN_FILL
                cell.font  = Font(bold=True, name="Calibri", size=10)

            # Override for thickness column → soft red
            if thickness_col and col_idx_ == thickness_col:
                cell.fill = THICK_FILL
                cell.font  = Font(bold=True, name="Calibri", size=10)

    # ── Column widths ─────────────────────────────────────────────────────────
    for col_num in range(1, ws.max_column + 1):
        col_letter = get_column_letter(col_num)
        header_val = ws.cell(1, col_num).value or ""
        if str(header_val).startswith("ray_"):
            ws.column_dimensions[col_letter].width = 13
        elif str(header_val) in ("axon_id", "axon_type", "num_inner_contours"):
            ws.column_dimensions[col_letter].width = 16
        else:
            ws.column_dimensions[col_letter].width = 18

    ws.row_dimensions[1].height = 40

    # ── Freeze header row ─────────────────────────────────────────────────────
    ws.freeze_panes = "A2"

    # ── Summary sheet ─────────────────────────────────────────────────────────
    ws_summary = wb.create_sheet("Summary")

    mature_df   = df[df["axon_type"] == "mature"]
    regrowth_df = df[df["axon_type"] == "regrowth_cluster"]

    summary_rows = [
        ("Image",                image_name),
        ("Total axons",          len(df)),
        ("Mature axons",         len(mature_df)),
        ("Regrowth clusters",    len(regrowth_df)),
        ("", ""),
        ("--- Mature Axons (ray-based measurements) ---", ""),
        ("Mean outer radius (µm)",   round(mature_df["outer_radius_um"].mean(), 4)   if len(mature_df) else "N/A"),
        ("Mean inner radius (µm)",   round(mature_df["inner_radius_um"].mean(), 4)   if len(mature_df) else "N/A"),
        ("Mean chosen radius (µm)",  round(mature_df["chosen_radius_um"].mean(), 4)  if len(mature_df) else "N/A"),
        ("Mean thickness (µm)",      round(mature_df["thickness_um"].mean(), 4)      if len(mature_df) else "N/A"),
        ("Mean g-ratio",             round(mature_df["g_ratio"].mean(), 4)           if len(mature_df) else "N/A"),
        ("Mean diameter (µm)",       round(mature_df["diameter_um"].mean(), 4)       if len(mature_df) else "N/A"),
        ("", ""),
        ("--- Regrowth Clusters ---", ""),
        ("Mean outer radius (µm)",   round(regrowth_df["outer_radius_um"].mean(), 4) if len(regrowth_df) else "N/A"),
        ("Mean thickness (µm)",      round(regrowth_df["thickness_um"].mean(), 4)    if len(regrowth_df) else "N/A"),
    ]

    for r, (label, value) in enumerate(summary_rows, start=1):
        c1 = ws_summary.cell(r, 1, value=label)
        c2 = ws_summary.cell(r, 2, value=value)
        c1.font = Font(bold=True, name="Calibri", size=10)
        c2.alignment = Alignment(horizontal="center")
        if str(label).startswith("---"):
            c1.fill = HEADER_FILL
            c1.font = WHITE_FONT

    ws_summary.column_dimensions["A"].width = 38
    ws_summary.column_dimensions["B"].width = 20

    wb.save(excel_path)
    print(f"  ✔ Excel saved → {excel_path}")
    return df


# --------------------------
# Visualisations (CSV also kept as backup)
# --------------------------
def save_csv_and_plots(df, output_folder):
    csv_path = os.path.join(output_folder, "axon_measurements.csv")
    df.to_csv(csv_path, index=False)

    # Only plot numeric, non-ray columns to keep output manageable
    non_ray_cols = [c for c in df.select_dtypes(include=[np.number]).columns
                    if not c.startswith("ray_")]

    for col in non_ray_cols:
        vals = df[col].dropna()
        if len(vals) == 0:
            continue
        plt.figure(figsize=(6, 4))
        plt.hist(vals, bins=25, color="steelblue", edgecolor="white")
        plt.title(col)
        plt.tight_layout()
        plt.savefig(os.path.join(output_folder, f"hist_{col}.png"))
        plt.close()

    # Extra: ray-distance box plot for mature axons
    mature_df = df[df["axon_type"] == "mature"]
    ray_cols  = [c for c in df.columns if c.startswith("ray_")]
    if len(mature_df) > 0 and ray_cols:
        ray_data = mature_df[ray_cols].values
        plt.figure(figsize=(max(8, len(ray_cols) // 2), 5))
        plt.boxplot(ray_data, labels=[c.replace("ray_", "").replace("deg_um", "°")
                                      for c in ray_cols],
                    patch_artist=True,
                    boxprops=dict(facecolor="lightblue"),
                    medianprops=dict(color="red"))
        plt.xticks(rotation=90, fontsize=7)
        plt.title("Ray distances per direction (mature axons, µm)")
        plt.ylabel("Distance (µm)")
        plt.tight_layout()
        plt.savefig(os.path.join(output_folder, "ray_distances_boxplot.png"))
        plt.close()


# --------------------------
# Process entire folder
# --------------------------
def process_folder(input_folder, output_root):
    images = [
        f for f in os.listdir(input_folder)
        if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp"))
    ]

    if not images:
        print("⚠ No images found in input folder.")
        return

    for img_name in images:
        print("\n==============================")
        print(f" Processing: {img_name}")
        print("==============================")

        img_path           = os.path.join(input_folder, img_name)
        img_name_wo_ext    = os.path.splitext(img_name)[0]
        img_output_folder  = os.path.join(output_root, img_name_wo_ext)
        os.makedirs(img_output_folder, exist_ok=True)

        axon_data, output_image = process_image(img_path)

        if axon_data is None:
            continue

        df = save_to_excel(axon_data, img_output_folder, img_name_wo_ext)
        if df is not None:
            save_csv_and_plots(df, img_output_folder)

        out_img_path = os.path.join(img_output_folder, f"numbered_{img_name}")
        cv2.imwrite(out_img_path, output_image)
        print(f"  ✔ Annotated image → {out_img_path}")

    print("\n🎉 DONE! All images processed.")


# --------------------------
# RUN
# --------------------------
process_folder(input_folder, output_root)


 Processing: 1.png
  → Processed 192 axons total (168 mature, 24 regrowth).
  ✔ Excel saved → C:\Users\DELL\Downloads\20x_bf_05_all_thickness\1\1_axon_measurements.xlsx


C:\Users\DELL\AppData\Local\Temp\ipykernel_18084\4116836642.py:571: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(ray_data, labels=[c.replace("ray_", "").replace("deg_um", "°")


  ✔ Annotated image → C:\Users\DELL\Downloads\20x_bf_05_all_thickness\1\numbered_1.png

 Processing: 2.png
  → Processed 340 axons total (292 mature, 48 regrowth).
  ✔ Excel saved → C:\Users\DELL\Downloads\20x_bf_05_all_thickness\2\2_axon_measurements.xlsx


C:\Users\DELL\AppData\Local\Temp\ipykernel_18084\4116836642.py:571: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(ray_data, labels=[c.replace("ray_", "").replace("deg_um", "°")


  ✔ Annotated image → C:\Users\DELL\Downloads\20x_bf_05_all_thickness\2\numbered_2.png

 Processing: 3.png
  → Processed 535 axons total (489 mature, 46 regrowth).
  ✔ Excel saved → C:\Users\DELL\Downloads\20x_bf_05_all_thickness\3\3_axon_measurements.xlsx


...

  ✔ Annotated image → C:\Users\DELL\Downloads\20x_bf_05_all_thickness\3\numbered_3.png

 Processing: 4.png
  → Processed 432 axons total (395 mature, 37 regrowth).
  ✔ Excel saved → C:\Users\DELL\Downloads\20x_bf_05_all_thickness\4\4_axon_measurements.xlsx


C:\Users\DELL\AppData\Local\Temp\ipykernel_18084\4116836642.py:571: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(ray_data, labels=[c.replace("ray_", "").replace("deg_um", "°")


  ✔ Annotated image → C:\Users\DELL\Downloads\20x_bf_05_all_thickness\4\numbered_4.png

 Processing: 5.png
  → Processed 277 axons total (254 mature, 23 regrowth).
  ✔ Excel saved → C:\Users\DELL\Downloads\20x_bf_05_all_thickness\5\5_axon_measurements.xlsx


C:\Users\DELL\AppData\Local\Temp\ipykernel_18084\4116836642.py:571: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(ray_data, labels=[c.replace("ray_", "").replace("deg_um", "°")


  ✔ Annotated image → C:\Users\DELL\Downloads\20x_bf_05_all_thickness\5\numbered_5.png

🎉 DONE! All images processed.


In [ ]:
import cv2
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# --------------------------
# CONFIG: Give folder containing input images
# --------------------------
input_folder = r"D:\projects\neuro\all fascicles\fascicle images\set2_Image_01_20x_bf_02"
output_root = r"C:\Users\DELL\Downloads\20x_bf_02_thickness2"
os.makedirs(output_root, exist_ok=True)

# Scale factor (µm/px)
scale_factor = 0.136

# --------------------------
# Ray casting config
# --------------------------
N_RAYS = 36          # Number of rays fired (every 10 degrees)
RAY_STEP = 0.5       # Step size in pixels along each ray

# --------------------------
# Tunables
# --------------------------
TISSUE_OVERLAP_MIN = 0.50
MAX_MEAN_INTENSITY = 245

MIN_CIRCULARITY = 0.05
MIN_CONTOUR_AREA = 8
MAX_CONTOUR_AREA = 2000000
MIN_SOLIDITY = 0.15

# Precompute ray angle labels once (used for column headers)
RAY_ANGLES             = np.linspace(0, 360, N_RAYS, endpoint=False)
OUTER_RAY_COL_LABELS   = [f"outer_ray_{int(a):03d}deg_um"     for a in RAY_ANGLES]
INNER_RAY_COL_LABELS   = [f"inner_ray_{int(a):03d}deg_um"     for a in RAY_ANGLES]
THICK_RAY_COL_LABELS   = [f"thickness_ray_{int(a):03d}deg_um" for a in RAY_ANGLES]

# --------------------------
# Helpers
# --------------------------
def darken_and_sharpen(image):
    darkened = cv2.convertScaleAbs(image, alpha=1.5, beta=-20)
    sharpen_kernel = np.array([[0, -1, 0],
                               [-1, 5, -1],
                               [0, -1, 0]])
    return cv2.filter2D(darkened, -1, sharpen_kernel)


def build_tissue_mask(full_image):
    hsv = cv2.cvtColor(full_image, cv2.COLOR_BGR2HSV)
    H, S, V = cv2.split(hsv)
    sat_mask = cv2.inRange(S, 15, 255)
    not_too_bright = cv2.inRange(V, 0, 250)
    mask = cv2.bitwise_and(sat_mask, not_too_bright)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k, iterations=2)
    return mask


def contour_overlap_ratio(contour, mask):
    c_mask = np.zeros(mask.shape, dtype=np.uint8)
    cv2.drawContours(c_mask, [contour], -1, 255, -1)
    inter = cv2.bitwise_and(c_mask, mask)
    area_c = cv2.countNonZero(c_mask)
    area_i = cv2.countNonZero(inter)
    return area_i / float(area_c) if area_c > 0 else 0.0


def mean_intensity_in_contour(gray, contour):
    m = np.zeros(gray.shape, dtype=np.uint8)
    cv2.drawContours(m, [contour], -1, 255, -1)
    return cv2.mean(gray, mask=m)[0]


def circularity(contour):
    a = cv2.contourArea(contour)
    p = cv2.arcLength(contour, True)
    return 4 * np.pi * a / (p * p + 1e-6) if p > 0 else 0


# --------------------------
# Ray Casting
# --------------------------
def cast_rays_to_boundary(contour, cx, cy, n_rays=N_RAYS, step=RAY_STEP):
    """
    Fire n_rays from (cx, cy) outward in equally spaced directions.
    For each ray, walks outward until it exits the filled contour.

    Returns:
        ray_distances_px : np.array of shape (n_rays,), distances in pixels
        ray_angles_deg   : np.array of shape (n_rays,), corresponding angles
    """
    # Build a local binary mask big enough to contain the contour
    x_b, y_b, w_b, h_b = cv2.boundingRect(contour)
    pad = 20
    max_r = int(max(w_b, h_b) + pad)

    # Shift contour so that (cx, cy) maps to (max_r, max_r) in local coords
    local_cx = max_r
    local_cy = max_r
    shifted = contour - np.array([[int(cx) - max_r, int(cy) - max_r]])

    mask_size = max_r * 2 + 2
    local_mask = np.zeros((mask_size, mask_size), dtype=np.uint8)
    cv2.drawContours(local_mask, [shifted], -1, 255, -1)

    angles = np.linspace(0, 2 * np.pi, n_rays, endpoint=False)
    ray_distances_px = np.zeros(n_rays)

    for i, angle in enumerate(angles):
        cos_a = np.cos(angle)
        sin_a = np.sin(angle)

        # Walk outward along this direction
        r_vals = np.arange(0, max_r, step)
        xs = (local_cx + r_vals * cos_a).astype(int)
        ys = (local_cy + r_vals * sin_a).astype(int)

        # Clip to mask bounds
        valid = (xs >= 0) & (xs < mask_size) & (ys >= 0) & (ys < mask_size)
        xs_v = xs[valid]
        ys_v = ys[valid]
        r_v  = r_vals[valid]

        if len(xs_v) == 0:
            ray_distances_px[i] = 0
            continue

        pixel_vals = local_mask[ys_v, xs_v]

        # Find first index where we go from inside (255) to outside (0)
        inside_idx = np.where(pixel_vals == 255)[0]
        if len(inside_idx) == 0:
            # centre not inside contour – unusual, fall back to 0
            ray_distances_px[i] = 0
            continue

        outside_after = np.where((pixel_vals == 0) & (np.arange(len(pixel_vals)) > inside_idx[0]))[0]
        if len(outside_after) > 0:
            ray_distances_px[i] = r_v[outside_after[0]]
        else:
            # ray never exited – use the last valid r
            ray_distances_px[i] = r_v[-1]

    return ray_distances_px, np.degrees(angles)


def draw_rays_on_image(output_image, cx, cy, ray_distances_px, ray_angles_deg, chosen_idx, x_offset, y_offset):
    """Draw all rays in cyan, the shortest (chosen) ray in red."""
    angles_rad = np.deg2rad(ray_angles_deg)
    for i, (r, ang) in enumerate(zip(ray_distances_px, angles_rad)):
        end_x = int(cx + r * np.cos(ang))
        end_y = int(cy + r * np.sin(ang))
        color = (0, 0, 255) if i == chosen_idx else (200, 200, 0)  # red vs dark-cyan
        cv2.line(output_image,
                 (int(cx + x_offset), int(cy + y_offset)),
                 (end_x + x_offset, end_y + y_offset),
                 color, 1)


# --------------------------
# Main patch processing
# --------------------------
def process_patch(patch, mask_patch, x_offset, y_offset, axon_data, object_counter, output_image, axons_image):

    patch = darken_and_sharpen(patch)
    gray = cv2.cvtColor(patch, cv2.COLOR_BGR2GRAY)

    smooth = cv2.bilateralFilter(gray, 5, 20, 20)

    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    enhanced = clahe.apply(smooth)

    blurred = cv2.GaussianBlur(enhanced, (3, 3), 0)
    _, otsu = cv2.threshold(blurred, 0, 255,
                            cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    adaptive = cv2.adaptiveThreshold(
        blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, 35, 2
    )

    thresh = cv2.bitwise_or(otsu, adaptive)

    kernel = np.ones((3, 3), np.uint8)
    opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=1)
    sure_bg = cv2.dilate(opening, kernel, iterations=3)

    dist_transform = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
    if dist_transform.max() > 0:
        _, sure_fg = cv2.threshold(dist_transform,
                                   0.4 * dist_transform.max(), 255, 0)
    else:
        sure_fg = np.zeros_like(opening)
    sure_fg = np.uint8(sure_fg)

    unknown = cv2.subtract(sure_bg, sure_fg)
    num_markers, markers = cv2.connectedComponents(sure_fg)
    markers = markers + 1
    markers[unknown == 255] = 0

    wshed_input = cv2.cvtColor(enhanced, cv2.COLOR_GRAY2BGR)
    markers = cv2.watershed(wshed_input, markers)

    cleaned = thresh.copy()
    cleaned[markers == -1] = 0

    contours, hierarchy = cv2.findContours(
        cleaned, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE
    )
    if hierarchy is None:
        return object_counter
    hierarchy = hierarchy[0]

    for i, contour in enumerate(contours):

        if hierarchy[i][3] != -1:
            continue

        area = cv2.contourArea(contour)
        if area < MIN_CONTOUR_AREA or area > MAX_CONTOUR_AREA:
            continue

        overlap = contour_overlap_ratio(contour, mask_patch)
        if overlap < TISSUE_OVERLAP_MIN:
            continue

        hull = cv2.convexHull(contour)
        solidity = area / (cv2.contourArea(hull) + 1e-6)
        circ = circularity(contour)
        if solidity < MIN_SOLIDITY or circ < MIN_CIRCULARITY:
            continue

        # Inner contours
        inner_children = []
        for j, child in enumerate(contours):
            if hierarchy[j][3] == i and cv2.contourArea(child) > 30:
                inner_children.append(child)

        if len(inner_children) == 0:
            continue

        axon_type = "mature" if len(inner_children) <= 2 else "regrowth_cluster"

        (x_outer, y_outer), outer_radius_enc = cv2.minEnclosingCircle(contour)
        contour_shifted = contour + np.array([x_offset, y_offset])
        x_outer_abs = x_outer + x_offset
        y_outer_abs = y_outer + y_offset

        outer_area_px  = cv2.contourArea(contour)
        outer_area_um2 = outer_area_px * (scale_factor ** 2)

        outer_color = (255, 0, 0) if axon_type == "mature" else (0, 255, 255)
        cv2.drawContours(output_image, [contour_shifted], -1, outer_color, 1)
        cv2.drawContours(axons_image,  [contour_shifted], -1, outer_color, 1)

        inner_radii = []
        inner_areas  = []
        for inner_c in inner_children:
            (_, _), ir = cv2.minEnclosingCircle(inner_c)
            inner_shifted = inner_c + np.array([x_offset, y_offset])
            cv2.drawContours(output_image, [inner_shifted], -1, (0, 255, 0), 1)
            cv2.drawContours(axons_image,  [inner_shifted], -1, (0, 255, 0), 1)
            inner_radii.append(ir)
            inner_areas.append(cv2.contourArea(inner_c))

        inner_radius   = max(inner_radii)
        inner_area_px  = max(inner_areas)
        inner_area_um2 = inner_area_px * (scale_factor ** 2)

        # --------------------------------------------------
        # RAY CASTING  (mature axons only)
        # Cast rays from the SAME centre for both outer and inner contours,
        # compute per-direction thickness, pick the minimum positive value.
        # --------------------------------------------------
        outer_ray_dict  = {col: np.nan for col in OUTER_RAY_COL_LABELS}
        inner_ray_dict  = {col: np.nan for col in INNER_RAY_COL_LABELS}
        thick_ray_dict  = {col: np.nan for col in THICK_RAY_COL_LABELS}

        outer_radius_um  = outer_radius_enc * scale_factor
        inner_radius_um  = inner_radius      * scale_factor
        chosen_outer_um  = outer_radius_um   # fallback = enclosing circle
        chosen_inner_um  = inner_radius_um
        chosen_thick_um  = max(outer_radius_um - inner_radius_um, 0.0)
        chosen_ray_angle = np.nan
        chosen_idx       = -1

        if axon_type == "mature":
            # ── Outer contour rays ──────────────────────────────────────────
            outer_rays_px, ray_angles_deg = cast_rays_to_boundary(
                contour, x_outer, y_outer, n_rays=N_RAYS, step=RAY_STEP
            )
            outer_rays_um = outer_rays_px * scale_factor

            # ── Inner contour rays (largest inner child, same origin) ───────
            largest_inner_c = inner_children[np.argmax(inner_areas)]
            inner_rays_px, _ = cast_rays_to_boundary(
                largest_inner_c, x_outer, y_outer, n_rays=N_RAYS, step=RAY_STEP
            )
            inner_rays_um = inner_rays_px * scale_factor

            # ── Per-direction thickness ──────────────────────────────────────
            thick_per_dir = outer_rays_um - inner_rays_um   # shape (N_RAYS,)

            # Store all per-direction values
            for o_col, i_col, t_col, o_val, i_val, t_val in zip(
                OUTER_RAY_COL_LABELS, INNER_RAY_COL_LABELS, THICK_RAY_COL_LABELS,
                outer_rays_um, inner_rays_um, thick_per_dir
            ):
                outer_ray_dict[o_col] = round(float(o_val), 4)
                inner_ray_dict[i_col] = round(float(i_val), 4)
                thick_ray_dict[t_col] = round(float(t_val), 4)

            # ── Pick direction with minimum POSITIVE thickness ───────────────
            positive_mask = thick_per_dir > 0
            if positive_mask.any():
                # among valid directions only
                valid_thick = np.where(positive_mask, thick_per_dir, np.inf)
                chosen_idx       = int(np.argmin(valid_thick))
                chosen_outer_um  = float(outer_rays_um[chosen_idx])
                chosen_inner_um  = float(inner_rays_um[chosen_idx])
                chosen_thick_um  = float(thick_per_dir[chosen_idx])
                chosen_ray_angle = float(ray_angles_deg[chosen_idx])
            else:
                # fallback – use enclosing-circle values
                chosen_thick_um  = max(outer_radius_um - inner_radius_um, 0.0)

            # Draw rays on output image
            draw_rays_on_image(
                output_image,
                x_outer, y_outer,
                outer_rays_px, ray_angles_deg,
                chosen_idx,
                x_offset, y_offset
            )

        thickness_um = chosen_thick_um

        g_ratio    = chosen_inner_um / (chosen_outer_um + 1e-6)
        area_ratio = inner_area_um2 / (outer_area_um2 + 1e-6)

        # Base record
        record = {
            "axon_id"              : object_counter,
            "axon_type"            : axon_type,
            "num_inner_contours"   : len(inner_children),
            "center_x_px"          : round(float(x_outer_abs), 2),
            "center_y_px"          : round(float(y_outer_abs), 2),
            # enclosing-circle radii (reference)
            "outer_radius_enc_um"  : round(outer_radius_um, 4),
            "inner_radius_enc_um"  : round(inner_radius_um, 4),
            # ray-chosen values
            "chosen_ray_angle_deg" : round(chosen_ray_angle, 2) if axon_type == "mature" else np.nan,
            "chosen_outer_ray_um"  : round(chosen_outer_um, 4),
            "chosen_inner_ray_um"  : round(chosen_inner_um, 4),
            "thickness_um"         : round(thickness_um, 4),
            "diameter_um"          : round(2 * chosen_outer_um, 4),
            "outer_area_um2"       : round(outer_area_um2, 4),
            "inner_area_um2"       : round(inner_area_um2, 4),
            "area_ratio"           : round(area_ratio, 4),
            "g_ratio"              : round(g_ratio, 4),
        }

        # Merge per-direction ray columns (only populated for mature)
        record.update(outer_ray_dict)
        record.update(inner_ray_dict)
        record.update(thick_ray_dict)

        axon_data.append(record)

        cv2.putText(output_image, str(object_counter),
                    (int(x_outer_abs), int(y_outer_abs)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
        cv2.putText(axons_image, str(object_counter),
                    (int(x_outer_abs), int(y_outer_abs)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)

        object_counter += 1

    return object_counter


# --------------------------
# Whole-image processing
# --------------------------
def process_image(image_path, patch_size=1024):
    image = cv2.imread(image_path)
    if image is None:
        print(f"❌ Cannot read image: {image_path}")
        return None, None

    tissue_mask_full = build_tissue_mask(image)
    h, w = image.shape[:2]

    output_image  = image.copy()
    axons_image   = image.copy()   # clean copy – contours only, no rays
    axon_data     = []
    object_counter = 1

    for y in range(0, h, patch_size):
        for x in range(0, w, patch_size):
            patch      = image[y:y + patch_size, x:x + patch_size]
            mask_patch = tissue_mask_full[y:y + patch_size, x:x + patch_size]

            object_counter = process_patch(
                patch, mask_patch, x, y, axon_data, object_counter, output_image, axons_image
            )

    print(f"  → Processed {len(axon_data)} axons total "
          f"({sum(1 for a in axon_data if a['axon_type'] == 'mature')} mature, "
          f"{sum(1 for a in axon_data if a['axon_type'] == 'regrowth_cluster')} regrowth).")
    return axon_data, output_image, axons_image


# --------------------------
# Excel export (styled)
# --------------------------
def save_to_excel(axon_data, output_folder, image_name):
    if not axon_data:
        print("  ⚠ No axon data to save.")
        return None

    df = pd.DataFrame(axon_data)

    # ── Column order ──────────────────────────────────────────────────────────
    base_cols = [
        "axon_id", "axon_type", "num_inner_contours",
        "center_x_px", "center_y_px",
        # enclosing-circle reference radii
        "outer_radius_enc_um", "inner_radius_enc_um",
        # per-direction outer rays
        *OUTER_RAY_COL_LABELS,
        # per-direction inner rays
        *INNER_RAY_COL_LABELS,
        # per-direction thickness (outer - inner per ray)
        *THICK_RAY_COL_LABELS,
        # chosen minimum-thickness direction summary
        "chosen_ray_angle_deg", "chosen_outer_ray_um", "chosen_inner_ray_um",
        "thickness_um", "diameter_um",
        "outer_area_um2", "inner_area_um2",
        "area_ratio", "g_ratio",
    ]
    # keep only columns that actually exist in df
    ordered_cols = [c for c in base_cols if c in df.columns]
    df = df[ordered_cols]

    excel_path = os.path.join(output_folder, f"{image_name}_axon_measurements.xlsx")
    df.to_excel(excel_path, index=False, sheet_name="Axon Measurements")

    # ── Styling with openpyxl ─────────────────────────────────────────────────
    wb = openpyxl.load_workbook(excel_path)
    ws = wb.active

    # Colour palettes
    HEADER_FILL    = PatternFill("solid", fgColor="1F4E79")   # dark blue
    OUTER_RAY_FILL = PatternFill("solid", fgColor="E8F4FD")   # light blue  – outer rays
    INNER_RAY_FILL = PatternFill("solid", fgColor="EAF4E8")   # light green – inner rays
    THICK_RAY_FILL = PatternFill("solid", fgColor="FFF2CC")   # light yellow – thickness rays
    MATURE_FILL    = PatternFill("solid", fgColor="D9F0D3")   # green  – mature rows
    REGROWTH_FILL  = PatternFill("solid", fgColor="FCE4D6")   # orange – regrowth rows
    CHOSEN_FILL    = PatternFill("solid", fgColor="FFD700")   # gold   – chosen ray summary cols
    THICK_FILL     = PatternFill("solid", fgColor="F4CCCC")   # soft red – final thickness col

    WHITE_FONT     = Font(color="FFFFFF", bold=True, name="Calibri", size=10)
    HEADER_BORDER  = Border(bottom=Side(style="medium", color="FFFFFF"))
    CELL_BORDER    = Border(bottom=Side(style="thin",   color="D0D0D0"))

    # Identify column indices by name
    col_names = [ws.cell(1, c).value for c in range(1, ws.max_column + 1)]

    def col_idx(name):
        try:    return col_names.index(name) + 1
        except: return None

    outer_ray_col_indices = [col_idx(c) for c in OUTER_RAY_COL_LABELS if col_idx(c)]
    inner_ray_col_indices = [col_idx(c) for c in INNER_RAY_COL_LABELS if col_idx(c)]
    thick_ray_col_indices = [col_idx(c) for c in THICK_RAY_COL_LABELS if col_idx(c)]
    chosen_cols           = {col_idx(c) for c in
                             ("chosen_ray_angle_deg", "chosen_outer_ray_um", "chosen_inner_ray_um")
                             if col_idx(c)}
    thickness_col         = col_idx("thickness_um")
    axon_type_col         = col_idx("axon_type")

    # ── Header row ────────────────────────────────────────────────────────────
    for cell in ws[1]:
        cell.fill      = HEADER_FILL
        cell.font      = WHITE_FONT
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border    = HEADER_BORDER

    # Special fills for key header cells
    for ci in outer_ray_col_indices:
        ws.cell(1, ci).fill = PatternFill("solid", fgColor="1565C0")  # darker blue
    for ci in inner_ray_col_indices:
        ws.cell(1, ci).fill = PatternFill("solid", fgColor="2E7D32")  # darker green
    for ci in thick_ray_col_indices:
        ws.cell(1, ci).fill = PatternFill("solid", fgColor="F57F17")  # darker amber
    for ci in chosen_cols:
        ws.cell(1, ci).fill = PatternFill("solid", fgColor="B8860B")  # dark gold
    if thickness_col:
        ws.cell(1, thickness_col).fill = PatternFill("solid", fgColor="8B0000")
        ws.cell(1, thickness_col).font = Font(color="FFFFFF", bold=True, name="Calibri", size=10)

    # ── Data rows ─────────────────────────────────────────────────────────────
    for row_idx in range(2, ws.max_row + 1):
        atype = ws.cell(row_idx, axon_type_col).value if axon_type_col else ""
        row_fill = MATURE_FILL if atype == "mature" else REGROWTH_FILL

        for col_idx_ in range(1, ws.max_column + 1):
            cell = ws.cell(row_idx, col_idx_)
            cell.border    = CELL_BORDER
            cell.alignment = Alignment(horizontal="center", vertical="center")
            cell.fill      = row_fill   # default

            if col_idx_ in outer_ray_col_indices:
                cell.fill = OUTER_RAY_FILL
            elif col_idx_ in inner_ray_col_indices:
                cell.fill = INNER_RAY_FILL
            elif col_idx_ in thick_ray_col_indices:
                cell.fill = THICK_RAY_FILL
            elif col_idx_ in chosen_cols:
                cell.fill = CHOSEN_FILL
                cell.font  = Font(bold=True, name="Calibri", size=10)
            elif thickness_col and col_idx_ == thickness_col:
                cell.fill = THICK_FILL
                cell.font  = Font(bold=True, name="Calibri", size=10)

    # ── Column widths ─────────────────────────────────────────────────────────
    for col_num in range(1, ws.max_column + 1):
        col_letter = get_column_letter(col_num)
        header_val = str(ws.cell(1, col_num).value or "")
        if header_val.startswith(("outer_ray_", "inner_ray_", "thickness_ray_")):
            ws.column_dimensions[col_letter].width = 14
        elif header_val in ("axon_id", "axon_type", "num_inner_contours"):
            ws.column_dimensions[col_letter].width = 16
        else:
            ws.column_dimensions[col_letter].width = 20

    ws.row_dimensions[1].height = 40

    # ── Freeze header row ─────────────────────────────────────────────────────
    ws.freeze_panes = "A2"

    # ── Summary sheet ─────────────────────────────────────────────────────────
    ws_summary = wb.create_sheet("Summary")

    mature_df   = df[df["axon_type"] == "mature"]
    regrowth_df = df[df["axon_type"] == "regrowth_cluster"]

    summary_rows = [
        ("Image",                image_name),
        ("Total axons",          len(df)),
        ("Mature axons",         len(mature_df)),
        ("Regrowth clusters",    len(regrowth_df)),
        ("", ""),
        ("--- Mature Axons (ray-based measurements) ---", ""),
        ("Mean outer radius enc. (µm)",  round(mature_df["outer_radius_enc_um"].mean(), 4)  if len(mature_df) else "N/A"),
        ("Mean inner radius enc. (µm)",  round(mature_df["inner_radius_enc_um"].mean(), 4)  if len(mature_df) else "N/A"),
        ("Mean chosen outer ray (µm)",   round(mature_df["chosen_outer_ray_um"].mean(), 4)  if len(mature_df) else "N/A"),
        ("Mean chosen inner ray (µm)",   round(mature_df["chosen_inner_ray_um"].mean(), 4)  if len(mature_df) else "N/A"),
        ("Mean thickness (µm)",          round(mature_df["thickness_um"].mean(), 4)          if len(mature_df) else "N/A"),
        ("Mean g-ratio",                 round(mature_df["g_ratio"].mean(), 4)               if len(mature_df) else "N/A"),
        ("Mean diameter (µm)",           round(mature_df["diameter_um"].mean(), 4)           if len(mature_df) else "N/A"),
        ("", ""),
        ("--- Regrowth Clusters ---", ""),
        ("Mean outer radius enc. (µm)",  round(regrowth_df["outer_radius_enc_um"].mean(), 4) if len(regrowth_df) else "N/A"),
        ("Mean thickness (µm)",          round(regrowth_df["thickness_um"].mean(), 4)        if len(regrowth_df) else "N/A"),
    ]

    for r, (label, value) in enumerate(summary_rows, start=1):
        c1 = ws_summary.cell(r, 1, value=label)
        c2 = ws_summary.cell(r, 2, value=value)
        c1.font = Font(bold=True, name="Calibri", size=10)
        c2.alignment = Alignment(horizontal="center")
        if str(label).startswith("---"):
            c1.fill = HEADER_FILL
            c1.font = WHITE_FONT

    ws_summary.column_dimensions["A"].width = 38
    ws_summary.column_dimensions["B"].width = 20

    wb.save(excel_path)
    print(f"  ✔ Excel saved → {excel_path}")
    return df


# --------------------------
# Visualisations — exactly 3 plots
# --------------------------
def save_plots(df, output_folder):
    mature_df = df[df["axon_type"] == "mature"]
    if len(mature_df) == 0:
        print("  ⚠ No mature axons — skipping plots.")
        return

    # ── Plot 1: Chosen-degree distribution (how many axons per degree bucket) ──
    # Shows outer chosen angle and inner chosen angle on the same bar chart.
    angle_vals = mature_df["chosen_ray_angle_deg"].dropna()
    if len(angle_vals) > 0:
        bins = np.arange(0, 361, 360 / N_RAYS)   # one bin per ray direction
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.hist(angle_vals, bins=bins, color="steelblue", edgecolor="white",
                label="Chosen angle (min-thickness direction)")
        ax.set_xlabel("Direction (degrees)")
        ax.set_ylabel("Number of mature axons")
        ax.set_title("Distribution of chosen ray direction across mature axons")
        ax.set_xlim(0, 360)
        ax.set_xticks(np.arange(0, 361, 30))
        ax.legend()
        fig.tight_layout()
        fig.savefig(os.path.join(output_folder, "plot_chosen_degree_distribution.png"), dpi=150)
        plt.close(fig)

    # ── Plot 2: Outer radius distribution across all mature axons ──────────────
    outer_vals = mature_df["chosen_outer_ray_um"].dropna()
    if len(outer_vals) > 0:
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.hist(outer_vals, bins=30, color="royalblue", edgecolor="white")
        ax.axvline(outer_vals.mean(), color="red", linestyle="--",
                   label=f"Mean = {outer_vals.mean():.2f} µm")
        ax.set_xlabel("Outer radius (µm)")
        ax.set_ylabel("Number of mature axons")
        ax.set_title("Outer radius distribution (mature axons)")
        ax.legend()
        fig.tight_layout()
        fig.savefig(os.path.join(output_folder, "plot_outer_radius_distribution.png"), dpi=150)
        plt.close(fig)

    # ── Plot 3: Inner radius distribution across all mature axons ──────────────
    inner_vals = mature_df["chosen_inner_ray_um"].dropna()
    if len(inner_vals) > 0:
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.hist(inner_vals, bins=30, color="seagreen", edgecolor="white")
        ax.axvline(inner_vals.mean(), color="red", linestyle="--",
                   label=f"Mean = {inner_vals.mean():.2f} µm")
        ax.set_xlabel("Inner radius (µm)")
        ax.set_ylabel("Number of mature axons")
        ax.set_title("Inner radius distribution (mature axons)")
        ax.legend()
        fig.tight_layout()
        fig.savefig(os.path.join(output_folder, "plot_inner_radius_distribution.png"), dpi=150)
        plt.close(fig)


# --------------------------
# Process entire folder
# --------------------------
def process_folder(input_folder, output_root):
    images = [
        f for f in os.listdir(input_folder)
        if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp"))
    ]

    if not images:
        print("⚠ No images found in input folder.")
        return

    for img_name in images:
        print("\n==============================")
        print(f" Processing: {img_name}")
        print("==============================")

        img_path           = os.path.join(input_folder, img_name)
        img_name_wo_ext    = os.path.splitext(img_name)[0]
        img_output_folder  = os.path.join(output_root, img_name_wo_ext)
        os.makedirs(img_output_folder, exist_ok=True)

        axon_data, output_image, axons_image = process_image(img_path)

        if axon_data is None:
            continue

        df = save_to_excel(axon_data, img_output_folder, img_name_wo_ext)
        if df is not None:
            save_plots(df, img_output_folder)

        out_img_path = os.path.join(img_output_folder, f"numbered_{img_name}")
        cv2.imwrite(out_img_path, output_image)
        print(f"  ✔ Annotated image (with rays) → {out_img_path}")

        axons_only_path = os.path.join(img_output_folder, f"axons_only_{img_name}")
        cv2.imwrite(axons_only_path, axons_image)
        print(f"  ✔ Axons-only image            → {axons_only_path}")

    print("\n🎉 DONE! All images processed.")


# --------------------------
# RUN
# --------------------------
process_folder(input_folder, output_root)


 Processing: 1.png
  → Processed 190 axons total (172 mature, 18 regrowth).
  ✔ Excel saved → C:\Users\DELL\Downloads\20x_bf_02_thickness2\1\1_axon_measurements.xlsx
  ✔ Annotated image (with rays) → C:\Users\DELL\Downloads\20x_bf_02_thickness2\1\numbered_1.png
  ✔ Axons-only image            → C:\Users\DELL\Downloads\20x_bf_02_thickness2\1\axons_only_1.png

 Processing: 2.png
  → Processed 12 axons total (12 mature, 0 regrowth).
  ✔ Excel saved → C:\Users\DELL\Downloads\20x_bf_02_thickness2\2\2_axon_measurements.xlsx
  ✔ Annotated image (with rays) → C:\Users\DELL\Downloads\20x_bf_02_thickness2\2\numbered_2.png
  ✔ Axons-only image            → C:\Users\DELL\Downloads\20x_bf_02_thickness2\2\axons_only_2.png

 Processing: 3.png
  → Processed 406 axons total (381 mature, 25 regrowth).
